# Prétraitement des Données

Ce notebook est utilisé pour prétraiter et tokenizer le dataset CNN/DailyMail.

In [ ]:
# Imports
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import json
import logging

# Hugging Face
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments

# Tokenization
import sentencepiece as spm
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print(" Imports effectués")
print(f"   • PyTorch: {torch.__version__}")
print(f"   • Transformers: {__import__('transformers').__version__}")
print(f"   • Datasets: {__import__('datasets').__version__}")
print(f"   • SentencePiece: {spm.__version__}")


In [3]:
# Chargement du dataset
print("=" * 60)
print(" CHARGEMENT DU DATASET")
print("=" * 60)

from datasets import load_dataset

print("\n Chargement depuis les fichiers Parquet...")

# Charger les trois splits
train_dataset = load_dataset("parquet", data_dir="data/raw", split="train")
val_dataset = load_dataset("parquet", data_dir="data/raw", split="validation")
test_dataset = load_dataset("parquet", data_dir="data/raw", split="test")

print("\n Dataset chargé avec succès !")

# Afficher les tailles
print("\n1. TAILLES DES SPLITS:")
print(f"   • Train:      {len(train_dataset):>10,} exemples")
print(f"   • Validation: {len(val_dataset):>10,} exemples")
print(f"   • Test:       {len(test_dataset):>10,} exemples")
print(f"   • TOTAL:      {len(train_dataset) + len(val_dataset) + len(test_dataset):>10,} exemples")

# Afficher les colonnes
print("\n2. COLONNES DISPONIBLES:")
print(f"   {train_dataset.column_names}")

# Afficher un exemple
print("\n3. EXEMPLE (premier exemple du train set):")
example = train_dataset[0]
print(f"   Article (premiers 200 caractères):")
print(f"   '{example['article'][:200]}...'")
print(f"\n   Résumé:")
print(f"   '{example['highlights']}'")

print("\n" + "=" * 60)


 CHARGEMENT DU DATASET

 Chargement depuis les fichiers Parquet...

 Dataset chargé avec succès !

1. TAILLES DES SPLITS:
   • Train:         287,113 exemples
   • Validation:     13,368 exemples
   • Test:           11,490 exemples
   • TOTAL:         311,971 exemples

2. COLONNES DISPONIBLES:
   ['article', 'highlights', 'id']

3. EXEMPLE (premier exemple du train set):
   Article (premiers 200 caractères):
   'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on ...'

   Résumé:
   'Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .'



In [4]:
# Paramètres du projet
print("=" * 60)
print("  PARAMÈTRES DU PROJET")
print("=" * 60)

# ─────────────────────────────────────────
# PARAMÈTRES DE TOKENIZATION
# ─────────────────────────────────────────
MAX_INPUT_LENGTH = 512      # Longueur max des articles (en tokens)
MAX_TARGET_LENGTH = 64      # Longueur max des résumés (en tokens)
VOCAB_SIZE = 32000          # Taille du vocabulaire

# ─────────────────────────────────────────
# PARAMÈTRES DES SPLITS
# ─────────────────────────────────────────
RANDOM_SEED = 42
TRAIN_SPLITS = [0.1, 0.5, 1.0]  # 10%, 50%, 100% du train set

# ─────────────────────────────────────────
# CHEMINS
# ─────────────────────────────────────────
BASE_DIR = Path(".")
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
CACHE_DIR = BASE_DIR / "data" / "cache"

# Créer les dossiers
for dir_path in [PROCESSED_DIR / "t5", 
                 PROCESSED_DIR / "transformer", 
                 PROCESSED_DIR / "tokenizer"]:
    dir_path.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────
# TOKENS SPÉCIAUX POUR SENTENCEPIECE
# ─────────────────────────────────────────
SPECIAL_TOKENS = [
    "<pad>",      # Padding
    "<unk>",      # Unknown
    "<s>",        # Beginning of sentence
    "</s>",       # End of sentence
    "<summarize>" # Token spécial pour la tâche de résumé
]

# ─────────────────────────────────────────
# AFFICHAGE DES PARAMÈTRES
# ─────────────────────────────────────────
print("\n CONFIGURATION:")
print(f"   • MAX_INPUT_LENGTH:   {MAX_INPUT_LENGTH}")
print(f"   • MAX_TARGET_LENGTH:  {MAX_TARGET_LENGTH}")
print(f"   • VOCAB_SIZE:         {VOCAB_SIZE}")
print(f"   • RANDOM_SEED:        {RANDOM_SEED}")
print(f"   • TRAIN_SPLITS:       {TRAIN_SPLITS}")
print(f"   • SPECIAL_TOKENS:      {len(SPECIAL_TOKENS)} tokens")

print("\n CHEMINS:")
print(f"   • RAW_DATA_DIR:      {RAW_DATA_DIR}")
print(f"   • PROCESSED_DIR:     {PROCESSED_DIR}")
print(f"   • CACHE_DIR:         {CACHE_DIR}")

print("\n Dossiers créés:")
print(f"   ✓ {PROCESSED_DIR / 't5'}")
print(f"   ✓ {PROCESSED_DIR / 'transformer'}")
print(f"   ✓ {PROCESSED_DIR / 'tokenizer'}")

print("\n" + "=" * 60)


  PARAMÈTRES DU PROJET

 CONFIGURATION:
   • MAX_INPUT_LENGTH:   512
   • MAX_TARGET_LENGTH:  64
   • VOCAB_SIZE:         32000
   • RANDOM_SEED:        42
   • TRAIN_SPLITS:       [0.1, 0.5, 1.0]
   • SPECIAL_TOKENS:      5 tokens

 CHEMINS:
   • RAW_DATA_DIR:      data\raw
   • PROCESSED_DIR:     data\processed
   • CACHE_DIR:         data\cache

 Dossiers créés:
   ✓ data\processed\t5
   ✓ data\processed\transformer
   ✓ data\processed\tokenizer



In [5]:
# Nettoyage léger des textes
print("=" * 60)
print(" NETTOYAGE LÉGER DES TEXTES")
print("=" * 60)

import re
import unicodedata

def clean_text(text):
    """
    Nettoyage léger d'un texte.
    
    Actions:
    - Normalisation Unicode (NFC)
    - Suppression des espaces multiples
    - Suppression des lignes vides multiples
    - Suppression des espaces en début/fin de lignes
    
    NE FAIT PAS:
    - Pas de lowercase (conserve la casse)
    - Pas de suppression de ponctuation
    - Pas de suppression de stop words
    """
    if not isinstance(text, str):
        return ""
    
    # Normalisation Unicode (NFC = forme canonique)
    text = unicodedata.normalize('NFC', text)
    
    # Supprimer les espaces multiples (remplacer par un seul espace)
    text = re.sub(r' +', ' ', text)
    
    # Supprimer les tabulations multiples
    text = re.sub(r'\t+', ' ', text)
    
    # Supprimer les lignes vides multiples (garder une seule ligne vide)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    
    # Supprimer les espaces en début/fin de chaque ligne
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    
    # Supprimer les espaces en début et fin du texte
    text = text.strip()
    
    return text

# Test de la fonction
print("\n1. TEST DE LA FONCTION DE NETTOYAGE:")
print("-" * 60)

test_text = """   
   
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him.


   
   
The actor, who has played the boy wizard since he was 11, will have access to his £20m ($41.1m) fortune.
"""

print("TEXTE ORIGINAL:")
print(repr(test_text))
print("\nTEXTE NETTOYÉ:")
print(repr(clean_text(test_text)))

# Test avec un deuxième exemple
test_text2 = "  Multiple    spaces   here\n\n\n\nAnd  empty  lines  "
print("\n" + "-" * 60)
print("DEUXIÈME TEST:")
print("ORIGINAL:", repr(test_text2))
print("NETTOYÉ: ", repr(clean_text(test_text2)))

print("\n Fonction de nettoyage définie !")


 NETTOYAGE LÉGER DES TEXTES

1. TEST DE LA FONCTION DE NETTOYAGE:
------------------------------------------------------------
TEXTE ORIGINAL:
"   \n\nLONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him.\n\n\n\n\nThe actor, who has played the boy wizard since he was 11, will have access to his £20m ($41.1m) fortune.\n"

TEXTE NETTOYÉ:
"LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him.\n\nThe actor, who has played the boy wizard since he was 11, will have access to his £20m ($41.1m) fortune."

------------------------------------------------------------
DEUXIÈME TEST:
ORIGINAL: '  Multiple    spaces   here\n\n\n\nAnd  empty  lines  '
NETTOYÉ:  'Multiple spaces here\n\nAnd empty lines'


In [6]:
# Application du nettoyage et création des splits
print("=" * 60)
print("  NETTOYAGE ET CRÉATION DES SPLITS")
print("=" * 60)

import re
import unicodedata
from datasets import Dataset

# ─────────────────────────────────────────
# FONCTION DE NETTOYAGE
# ─────────────────────────────────────────
def clean_text(text):
    """
    Nettoyage léger d'un texte.
    - Normalisation Unicode (NFC)
    - Suppression des espaces multiples
    - Suppression des lignes vides multiples
    - NE TOUCHE PAS à la casse ou la ponctuation
    """
    if not isinstance(text, str):
        return ""
    
    # Normalisation Unicode (NFC)
    text = unicodedata.normalize('NFC', text)
    
    # Supprimer les espaces multiples
    text = re.sub(r' +', ' ', text)
    
    # Supprimer les tabulations multiples
    text = re.sub(r'\t+', ' ', text)
    
    # Supprimer les lignes vides multiples
    text = re.sub(r'\n\s*\n', '\n\n', text)
    
    # Supprimer les espaces en début/fin de chaque ligne
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    
    # Supprimer les espaces en début et fin du texte
    text = text.strip()
    
    return text

# ─────────────────────────────────────────
# NETTOYAGE DU DATASET
# ─────────────────────────────────────────
print("\n1. NETTOYAGE DU DATASET...")
print("    Cela peut prendre 2-3 minutes...")

def clean_dataset(examples):
    """Appliquer le nettoyage à un batch d'exemples."""
    examples['article'] = [clean_text(article) for article in examples['article']]
    examples['highlights'] = [clean_text(summary) for summary in examples['highlights']]
    return examples

# Nettoyer le train set
train_dataset_clean = train_dataset.map(
    clean_dataset,
    batched=True,
    batch_size=1000,
    desc="Nettoyage du train set"
)

# Nettoyer le validation set
val_dataset_clean = val_dataset.map(
    clean_dataset,
    batched=True,
    batch_size=1000,
    desc="Nettoyage du validation set"
)

# Nettoyer le test set
test_dataset_clean = test_dataset.map(
    clean_dataset,
    batched=True,
    batch_size=1000,
    desc="Nettoyage du test set"
)

print(" Nettoyage terminé !")

# ─────────────────────────────────────────
# CRÉATION DES SPLITS 10% / 50% / 100%
# ─────────────────────────────────────────
print("\n2. CRÉATION DES SPLITS D'ENTRAÎNEMENT...")

# Calculer les tailles
train_sizes = {
    "10": int(len(train_dataset_clean) * 0.1),   # 10%
    "50": int(len(train_dataset_clean) * 0.5),   # 50%
    "100": len(train_dataset_clean)               # 100%
}

print(f"   • Train 10%:  {train_sizes['10']:>10,} exemples")
print(f"   • Train 50%:  {train_sizes['50']:>10,} exemples")
print(f"   • Train 100%: {train_sizes['100']:>10,} exemples")

# Créer les splits avec seed fixe pour la reproductibilité
np.random.seed(RANDOM_SEED)

# Indices aléatoires pour la sélection
indices_10 = np.random.choice(len(train_dataset_clean), size=train_sizes['10'], replace=False)
indices_50 = np.random.choice(len(train_dataset_clean), size=train_sizes['50'], replace=False)

# Créer les datasets
train_10 = train_dataset_clean.select(indices_10)
train_50 = train_dataset_clean.select(indices_50)
train_100 = train_dataset_clean

print("\n Splits créés !")

# ─────────────────────────────────────────
# VÉRIFICATION
# ─────────────────────────────────────────
print("\n3. VÉRIFICATION DES SPLITS:")
print("-" * 60)

splits = {
    "train_10": train_10,
    "train_50": train_50,
    "train_100": train_100,
    "validation": val_dataset_clean,
    "test": test_dataset_clean
}

for name, ds in splits.items():
    print(f"   • {name:15s}: {len(ds):>10,} exemples")

# Vérifier que les splits sont bien disjoints
print("\n4. VÉRIFICATION DE LA DISJOINTURE:")
print(f"   • train_10 ∩ train_50: {len(set(train_10.column_names) & set(train_50.column_names))} colonnes communes")

print("\n Vérification terminée !")
print("\n" + "=" * 60)


  NETTOYAGE ET CRÉATION DES SPLITS

1. NETTOYAGE DU DATASET...
    Cela peut prendre 2-3 minutes...
 Nettoyage terminé !

2. CRÉATION DES SPLITS D'ENTRAÎNEMENT...
   • Train 10%:      28,711 exemples
   • Train 50%:     143,556 exemples
   • Train 100%:    287,113 exemples

 Splits créés !

3. VÉRIFICATION DES SPLITS:
------------------------------------------------------------
   • train_10       :     28,711 exemples
   • train_50       :    143,556 exemples
   • train_100      :    287,113 exemples
   • validation     :     13,368 exemples
   • test           :     11,490 exemples

4. VÉRIFICATION DE LA DISJOINTURE:
   • train_10 ∩ train_50: 3 colonnes communes

 Vérification terminée !



In [7]:
# Vérification rapide des splits
print("Splits en mémoire:")
print(f"  train_10:  {len(train_10):,} exemples")
print(f"  train_50:  {len(train_50):,} exemples")
print(f"  train_100: {len(train_100):,} exemples")


Splits en mémoire:
  train_10:  28,711 exemples
  train_50:  143,556 exemples
  train_100: 287,113 exemples


In [8]:
# Entraînement du tokenizer SentencePiece
print("=" * 60)
print(" ENTRAÎNEMENT DU TOKENIZER SENTENCEPIECE")
print("=" * 60)

import sentencepiece as spm
from pathlib import Path

# ─────────────────────────────────────────
# ÉTAPE 1 : PRÉPARER LE CORPUS
# ─────────────────────────────────────────
print("\n1. PRÉPARATION DU CORPUS...")
print("    Création d'un fichier texte avec tous les articles et résumés...")

# Créer le dossier cache s'il n'existe pas
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Créer un fichier temporaire pour l'entraînement
corpus_file = CACHE_DIR / "sentencepiece_corpus.txt"

# Écrire tous les articles et résumés dans un seul fichier
print("    Écriture du corpus (cela peut prendre 1-2 minutes)...")

with open(corpus_file, 'w', encoding='utf-8') as f:
    # Écrire les articles du train_100
    for i in range(len(train_100)):
        article = train_100[i]['article']
        f.write(article + '\n')
        
        # Écrire le résumé correspondant
        summary = train_100[i]['highlights']
        f.write(summary + '\n')

print(f"    Corpus créé: {corpus_file}")

# Vérifier la taille du fichier
corpus_size = corpus_file.stat().st_size / (1024 * 1024)  # MB
print(f"    Taille du corpus: {corpus_size:.2f} MB")


# ─────────────────────────────────────────
# ÉTAPE 2 : CONFIGURER L'ENTRAÎNEMENT
# ─────────────────────────────────────────
print("\n2. CONFIGURATION DE SENTENCEPIECE...")

# Chemin de sauvegarde du modèle
sp_model_path = PROCESSED_DIR / "tokenizer" / "sentencepiece"

# Paramètres d'entraînement
spm.SentencePieceTrainer.train(
    input=str(corpus_file),
    model_prefix=str(sp_model_path),
    vocab_size=VOCAB_SIZE,
    character_coverage=1.0,  # Couvrir tous les caractères
    model_type="bpe",  # Byte Pair Encoding
    pad_id=0,
    eos_id=1,
    unk_id=2,
    bos_id=3,
    pad_piece="<pad>",
    eos_piece="</s>",
    unk_piece="<unk>",
    bos_piece="<s>",
    user_defined_symbols=["<summarize>"],
    # normalization_rule_name ENLEVÉ (on l'a déjà fait dans clean_text)
    remove_extra_whitespaces=False,  # On a déjà nettoyé
    input_sentence_size=1000000,
    shuffle_input_sentence=True,
    seed_sentencepiece_size=RANDOM_SEED,
    hard_vocab_limit=False,
)

print(f"    Modèle SentencePiece entraîné !")
print(f"    Modèle sauvegardé dans: {sp_model_path}")


# ─────────────────────────────────────────
# ÉTAPE 3 : VÉRIFIER LE MODÈLE
# ─────────────────────────────────────────
print("\n3. VÉRIFICATION DU MODÈLE...")

# Charger le modèle entraîné
sp = spm.SentencePieceProcessor()
sp.load(str(sp_model_path) + ".model")

print(f"    Vocabulaire: {sp.get_piece_size():,} tokens")
print(f"    Token <pad>: {sp.pad_id()}")
print(f"    Token <unk>: {sp.unk_id()}")
print(f"    Token <s>: {sp.bos_id()}")
print(f"    Token </s>: {sp.eos_id()}")

# Test de tokenization
test_text = "summarize: Harry Potter star Daniel Radcliffe gets £20M fortune"
encoded = sp.encode(test_text, out_type=int)
decoded = sp.decode(encoded)

print(f"\n   TEST DE TOKENIZATION:")
print(f"   Texte original: {test_text}")
print(f"   Tokens: {encoded[:20]}...")  # Afficher les 20 premiers tokens
print(f"   Décode: {decoded}")

# Sauvegarder le vocabulaire en format texte
vocab_file = PROCESSED_DIR / "tokenizer" / "sentencepiece.vocab"
with open(vocab_file, 'w', encoding='utf-8') as f:
    for i in range(sp.get_piece_size()):
        piece = sp.id_to_piece(i)
        f.write(f"{piece}\n")

print(f"\n    Vocabulaire sauvegardé: {vocab_file}")

# ─────────────────────────────────────────
# ÉTAPE 4 : STATISTIQUES
# ─────────────────────────────────────────
print("\n4. STATISTIQUES DU TOKENIZER:")
print("-" * 60)

# Tester sur quelques exemples
test_sentences = [
    "summarize: " + train_100[0]['highlights'],
    train_100[100]['article'][:200],
    "The actor said he has no plans to fritter his cash away."
]

for i, sent in enumerate(test_sentences[:3]):
    tokens = sp.encode(sent, out_type=int)
    print(f"   Exemple {i+1}: {len(tokens)} tokens")

print("\n Entraînement du tokenizer terminé !")
print("\n" + "=" * 60)


 ENTRAÎNEMENT DU TOKENIZER SENTENCEPIECE

1. PRÉPARATION DU CORPUS...
    Création d'un fichier texte avec tous les articles et résumés...
    Écriture du corpus (cela peut prendre 1-2 minutes)...
    Corpus créé: data\cache\sentencepiece_corpus.txt
    Taille du corpus: 1190.78 MB

2. CONFIGURATION DE SENTENCEPIECE...
    Modèle SentencePiece entraîné !
    Modèle sauvegardé dans: data\processed\tokenizer\sentencepiece

3. VÉRIFICATION DU MODÈLE...
    Vocabulaire: 32,000 tokens
    Token <pad>: 0
    Token <unk>: 2
    Token <s>: 3
    Token </s>: 1

   TEST DE TOKENIZATION:
   Texte original: summarize: Harry Potter star Daniel Radcliffe gets £20M fortune
   Tokens: [1478, 2551, 1764, 31716, 3719, 10676, 631, 3229, 24521, 3725, 7690, 31704, 9054]...
   Décode: summarize: Harry Potter star Daniel Radcliffe gets £20M fortune

    Vocabulaire sauvegardé: data\processed\tokenizer\sentencepiece.vocab

4. STATISTIQUES DU TOKENIZER:
---------------------------------------------------------

In [9]:
# Préparation des données pour T5
print("=" * 60)
print(" PRÉPARATION DES DONNÉES POUR T5")
print("=" * 60)

from transformers import AutoTokenizer

# ─────────────────────────────────────────
# ÉTAPE 1 : CHARGER LE TOKENIZER T5
# ─────────────────────────────────────────
print("\n1. CHARGEMENT DU TOKENIZER T5...")

# Charger le tokenizer T5 pré-entraîné
t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")

print(" Tokenizer T5 chargé !")
print(f"   • Vocabulaire: {t5_tokenizer.vocab_size:,} tokens")
print(f"   • Modèle: t5-small")

# ─────────────────────────────────────────
# ÉTAPE 2 : FONCTION DE TOKENIZATION
# ─────────────────────────────────────────
print("\n2. FONCTION DE TOKENIZATION...")

def tokenize_for_t5(examples):
    """
    Tokenizer les articles et résumés pour T5.
    
    Pour T5, on ajoute le préfixe "summarize: " aux articles.
    """
    # Ajouter le préfixe "summarize: " aux articles
    inputs = ["summarize: " + article for article in examples['article']]
    
    # Tokenizer les inputs (articles)
    model_inputs = t5_tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True,
    )
    
    # Tokenizer les labels (résumés)
    labels = t5_tokenizer(
        examples['highlights'],
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True,
    )
    
    # Remplacer les padding tokens par -100 (ignorés par la loss)
    labels["input_ids"] = [
        [(label if label != t5_tokenizer.pad_token_id else -100) 
         for label in labels_example]
        for labels_example in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

print(" Fonction de tokenization définie !")

# ─────────────────────────────────────────
# ÉTAPE 3 : TOKENIZER LES SPLITS
# ─────────────────────────────────────────
print("\n3. TOKENIZATION DES SPLITS...")
print("    Cela peut prendre 2-3 minutes...")

# Tokenizer train_10
print("   • Tokenization de train_10...")
train_10_t5 = train_10.map(
    tokenize_for_t5,
    batched=True,
    batch_size=1000,
    desc="Tokenization train_10"
)

# Tokenizer train_50
print("   • Tokenization de train_50...")
train_50_t5 = train_50.map(
    tokenize_for_t5,
    batched=True,
    batch_size=1000,
    desc="Tokenization train_50"
)

# Tokenizer train_100
print("   • Tokenization de train_100...")
train_100_t5 = train_100.map(
    tokenize_for_t5,
    batched=True,
    batch_size=1000,
    desc="Tokenization train_100"
)

# Tokenizer validation
print("   • Tokenization de validation...")
val_t5 = val_dataset_clean.map(
    tokenize_for_t5,
    batched=True,
    batch_size=1000,
    desc="Tokenization validation"
)

# Tokenizer test
print("   • Tokenization de test...")
test_t5 = test_dataset_clean.map(
    tokenize_for_t5,
    batched=True,
    batch_size=1000,
    desc="Tokenization test"
)

print(" Tokenization T5 terminée !")

# ─────────────────────────────────────────
# ÉTAPE 4 : VÉRIFICATION
# ─────────────────────────────────────────
print("\n4. VÉRIFICATION...")
print("-" * 60)

# Vérifier les colonnes
print("Colonnes du dataset tokenizé:")
print(f"  {train_10_t5.column_names}")

# Afficher un exemple
example = train_10_t5[0]
print(f"\nExemple tokenizé:")
print(f"  • Input IDs shape: {len(example['input_ids'])}")
print(f"  • Attention mask shape: {len(example['attention_mask'])}")
print(f"  • Labels shape: {len(example['labels'])}")

# Décoder pour vérifier
decoded_input = t5_tokenizer.decode(example['input_ids'], skip_special_tokens=True)
decoded_label = t5_tokenizer.decode([t for t in example['labels'] if t != -100], skip_special_tokens=True)

print(f"\nVérification (décodage):")
print(f"  Input: {decoded_input[:100]}...")
print(f"  Label: {decoded_label[:100]}...")

print("\n Données T5 prêtes !")
print("\n" + "=" * 60)


 PRÉPARATION DES DONNÉES POUR T5

1. CHARGEMENT DU TOKENIZER T5...


 Tokenizer T5 chargé !
   • Vocabulaire: 32,100 tokens
   • Modèle: t5-small

2. FONCTION DE TOKENIZATION...
 Fonction de tokenization définie !

3. TOKENIZATION DES SPLITS...
    Cela peut prendre 2-3 minutes...
   • Tokenization de train_10...
   • Tokenization de train_50...
   • Tokenization de train_100...
   • Tokenization de validation...
   • Tokenization de test...
 Tokenization T5 terminée !

4. VÉRIFICATION...
------------------------------------------------------------
Colonnes du dataset tokenizé:
  ['article', 'highlights', 'id', 'input_ids', 'attention_mask', 'labels']

Exemple tokenizé:
  • Input IDs shape: 512
  • Attention mask shape: 512
  • Labels shape: 64

Vérification (décodage):
  Input: summarize: Nasa has warned of an impending asteroid pass - and says it will be the closest until 202...
  Label: 2004 BL86 will pass about three times the distance of Earth to the moon. Estimate that the asteroid ...

 Données T5 prêtes !



In [10]:
# Préparation des Données pour Transformer
print("=" * 60)
print(" PRÉPARATION DES DONNÉES POUR TRANSFORMER")
print("=" * 60)

import sentencepiece as spm
from pathlib import Path
import gc
import logging
from datasets import Dataset

logger = logging.getLogger(__name__)

# ─────────────────────────────────────────
# ÉTAPE 1 : CHARGER LE TOKENIZER SENTENCEPIECE
# ─────────────────────────────────────────
print("\n1. CHARGEMENT DU TOKENIZER SENTENCEPIECE...")

sp_model_path = PROCESSED_DIR / "tokenizer" / "sentencepiece.model"
sp = spm.SentencePieceProcessor()
sp.load(str(sp_model_path))

print(" Tokenizer SentencePiece chargé !")
print(f"   • Vocabulaire: {sp.get_piece_size():,} tokens")

# ─────────────────────────────────────────
# ÉTAPE 2 : FONCTION DE TOKENIZATION
# ─────────────────────────────────────────
print("\n2. FONCTION DE TOKENIZATION...")

def tokenize_for_transformer_batch(examples):
    """Tokenizer un batch d'exemples pour le Transformer."""
    input_ids = []
    attention_masks = []
    labels = []
    
    for article, summary in zip(examples['article'], examples['highlights']):
        # Tokenizer l'article avec préfixe
        article_with_prefix = "<summarize> " + article
        input_tokens = sp.encode(article_with_prefix, out_type=int)
        
        # Truncate ou pad
        if len(input_tokens) > MAX_INPUT_LENGTH:
            input_tokens = input_tokens[:MAX_INPUT_LENGTH]
        else:
            input_tokens = input_tokens + [0] * (MAX_INPUT_LENGTH - len(input_tokens))
        
        attention_mask = [1 if t != 0 else 0 for t in input_tokens]
        
        # Tokenizer le résumé avec EOS token
        label_tokens = sp.encode(summary, out_type=int)
        label_tokens = label_tokens + [sp.eos_id()]
        
        if len(label_tokens) > MAX_TARGET_LENGTH:
            label_tokens = label_tokens[:MAX_TARGET_LENGTH]
        else:
            label_tokens = label_tokens + [0] * (MAX_TARGET_LENGTH - len(label_tokens))
        
        input_ids.append(input_tokens)
        attention_masks.append(attention_mask)
        labels.append(label_tokens)
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    }

print(" Fonction de tokenization définie !")

# ─────────────────────────────────────────
# ÉTAPE 3 : TOKENIZER ET SAUVEGARDER (VERSION OPTIMISÉE)
# ─────────────────────────────────────────
print("\n3. TOKENIZATION ET SAUVEGARDE...")
print("    Utilisation de HuggingFace Datasets + sauvegarde par shards...")

# Créer le dossier de sortie
TRANSFORMER_DIR = PROCESSED_DIR / "transformer"
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)

def tokenize_and_save_optimized(dataset, split_name, batch_size=2000):
    """
    Tokenizer et sauvegarder un dataset de manière optimisée.
    
    Args:
        dataset: Dataset HuggingFace à tokenizer
        split_name: Nom du split (train_10, train_50, etc.)
        batch_size: Taille des batches de traitement
    """
    print(f"\n   • Traitement de {split_name}...")
    print(f"     Dataset: {len(dataset):,} exemples")
    
    # Tokenizer le dataset complet
    tokenized_dataset = dataset.map(
        tokenize_for_transformer_batch,
        batched=True,
        batch_size=batch_size,
        remove_columns=dataset.column_names,
        desc=f"  Tokenization {split_name}"
    )
    
    # Sauvegarder avec HuggingFace (format Arrow optimisé)
    output_path = TRANSFORMER_DIR / split_name
    print(f"     Sauvegarde dans {output_path}...")
    
    # Sauvegarder avec num_shards pour éviter la saturation
    num_shards = max(1, len(tokenized_dataset) // 10000)  # 1 shard par 10k exemples
    tokenized_dataset.save_to_disk(output_path, num_shards=num_shards)
    
    print(f"     ✓ {split_name} sauvegardé: {len(tokenized_dataset):,} exemples")
    print(f"     ✓ Shards: {num_shards}")
    
    # Calculer la taille totale
    total_size = sum(f.stat().st_size for f in output_path.glob("*.arrow") if f.is_file())
    print(f"     ✓ Taille: {total_size / (1024*1024):.2f} MB")
    
    # Libérer la mémoire
    del tokenized_dataset
    gc.collect()
    
    print(f"     ✓ Mémoire libérée")

# Tokenizer et sauvegarder chaque split
try:
    train_10_transformer = tokenize_and_save_optimized(train_10, "train_10")
    print("  ✓ train_10 terminé")
    gc.collect()
    
    train_50_transformer = tokenize_and_save_optimized(train_50, "train_50")
    print("  ✓ train_50 terminé")
    gc.collect()
    
    train_100_transformer = tokenize_and_save_optimized(train_100, "train_100")
    print("  ✓ train_100 terminé")
    gc.collect()
    
    val_transformer = tokenize_and_save_optimized(val_dataset_clean, "validation")
    print("  ✓ validation terminé")
    gc.collect()
    
    test_transformer = tokenize_and_save_optimized(test_dataset_clean, "test")
    print("  ✓ test terminé")
    gc.collect()
    
    print("\n Tokenization et sauvegarde Transformer terminées !")
    
except Exception as e:
    print(f"\n ERREUR: {e}")
    import traceback
    traceback.print_exc()




 PRÉPARATION DES DONNÉES POUR TRANSFORMER

1. CHARGEMENT DU TOKENIZER SENTENCEPIECE...
 Tokenizer SentencePiece chargé !
   • Vocabulaire: 32,000 tokens

2. FONCTION DE TOKENIZATION...
 Fonction de tokenization définie !

3. TOKENIZATION ET SAUVEGARDE...
    Utilisation de HuggingFace Datasets + sauvegarde par shards...

   • Traitement de train_10...
     Dataset: 28,711 exemples


  Tokenization train_10:   0%|          | 0/28711 [00:00<?, ? examples/s]

     Sauvegarde dans data\processed\transformer\train_10...


Saving the dataset (0/2 shards):   0%|          | 0/28711 [00:00<?, ? examples/s]

     ✓ train_10 sauvegardé: 28,711 exemples
     ✓ Shards: 2
     ✓ Taille: 84.46 MB
     ✓ Mémoire libérée
  ✓ train_10 terminé

   • Traitement de train_50...
     Dataset: 143,556 exemples


...

  Tokenization test:   0%|          | 0/11490 [00:00<?, ? examples/s]

     Sauvegarde dans data\processed\transformer\test...


Saving the dataset (0/1 shards):   0%|          | 0/11490 [00:00<?, ? examples/s]

     ✓ test sauvegardé: 11,490 exemples
     ✓ Shards: 1
     ✓ Taille: 33.80 MB
     ✓ Mémoire libérée
  ✓ test terminé

 Tokenization et sauvegarde Transformer terminées !

4. VÉRIFICATION...
------------------------------------------------------------


AttributeError: 'Dataset' object has no attribute 'num_shards'

In [11]:
# ─────────────────────────────────────────
# ÉTAPE 4 : VÉRIFICATION
# ─────────────────────────────────────────
print("\n4. VÉRIFICATION...")
print("-" * 60)

from datasets import load_from_disk
import logging

logger = logging.getLogger(__name__)

# Vérifier les fichiers
for split_name in ["train_10", "train_50", "train_100", "validation", "test"]:
    output_path = TRANSFORMER_DIR / split_name
    
    if output_path.exists():
        try:
            # Charger le dataset
            ds = load_from_disk(output_path)
            
            # Calculer la taille totale
            total_size = sum(f.stat().st_size for f in output_path.glob("*.arrow") if f.is_file())
            
            # CORRECTION: Compter les shards manuellement
            num_shards = len(list(output_path.glob("*.arrow")))
            
            print(f"   ✓ {split_name:15s}: {len(ds):>10,} exemples | {total_size / (1024*1024):>8.2f} MB | {num_shards} shards")
            
            # Vérifier les colonnes
            print(f"     Colonnes: {ds.column_names}")
            
            # Afficher un exemple
            example = ds[0]
            print(f"     Input IDs: {len(example['input_ids'])} tokens")
            print(f"     Attention mask: {len(example['attention_mask'])} tokens")
            print(f"     Labels: {len(example['labels'])} tokens")
            
            # Décoder pour vérifier
            decoded_input = sp.decode([t for t in example['input_ids'] if t != 0])
            decoded_label = sp.decode([t for t in example['labels'] if t != 0])
            print(f"     Input: {decoded_input[:100]}...")
            print(f"     Label: {decoded_label[:100]}...")
            
            # Libérer la mémoire
            del ds
            gc.collect()
            
        except Exception as e:
            print(f"    Erreur lors de la vérification de {split_name}: {e}")
            logger.error(f"Erreur pour {split_name}: {e}", exc_info=True)
    else:
        print(f"    {split_name:15s}: N'existe pas!")

print("\n" + "=" * 60)
print(" Données Transformer prêtes et sauvegardées !")
print(f"\n Dossier: {TRANSFORMER_DIR}")
print("\n Structure des fichiers:")
for split_name in ["train_10", "train_50", "train_100", "validation", "test"]:
    output_path = TRANSFORMER_DIR / split_name
    if output_path.exists():
        shards = list(output_path.glob("*.arrow"))
        dataset_info = output_path / "dataset_info.json"
        print(f"  {split_name}/: {len(shards)} shards" + (", dataset_info.json" if dataset_info.exists() else ""))
print("=" * 60)



4. VÉRIFICATION...
------------------------------------------------------------
   ✓ train_10       :     28,711 exemples |    84.46 MB | 2 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: <summarize> Nasa has warned of an impending asteroid pass - and says it will be the closest until 20...
     Label: 2004 BL86 will pass about three times the distance of Earth to the moon . Estimate that the asteroid...
   ✓ train_50       :    143,556 exemples |   422.29 MB | 14 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: <summarize> (CNN) -- When convicted intelligence leaker Bradley Manning asked Thursday for hormone t...
     Label: Convicted intel leaker Bradley Manning has said he is female . The Army has refused to provide hormo...


Loading dataset from disk:   0%|          | 0/28 [00:00<?, ?it/s]

   ✓ train_100      :    287,113 exemples |   844.58 MB | 28 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: <summarize> LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a report...
     Label: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday . Young actor says he has...
   ✓ validation     :     13,368 exemples |    39.32 MB | 1 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: <summarize> (CNN)Share, and your gift will be multiplied. That may sound like an esoteric adage, but...
     Label: Zully Broussard decided to give a kidney to a stranger . A new computer program helped her donation ...
   ✓ test           :     11,490 exemples |    33.80 MB | 1 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']

In [12]:
# ============================================================
# PRÉPARATION DES DONNÉES POUR T5
# ============================================================

print("=" * 60)
print(" PRÉPARATION DES DONNÉES POUR T5")
print("=" * 60)

from transformers import AutoTokenizer
from pathlib import Path
import gc

# ─────────────────────────────────────────
# ÉTAPE 1 : CHARGER LE TOKENIZER T5
# ─────────────────────────────────────────
print("\n1. CHARGEMENT DU TOKENIZER T5...")

t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")

print(" Tokenizer T5 chargé !")
print(f"   • Vocabulaire: {len(t5_tokenizer):,} tokens")
print(f"   • Modèle: t5-small")

# ─────────────────────────────────────────
# ÉTAPE 2 : FONCTION DE TOKENIZATION
# ─────────────────────────────────────────
print("\n2. FONCTION DE TOKENIZATION...")

def tokenize_for_t5_batch(examples):
    """Tokenizer un batch d'exemples pour T5."""
    input_ids = []
    attention_masks = []
    labels = []
    
    for article, summary in zip(examples['article'], examples['highlights']):
        # T5 utilise un préfixe pour la tâche
        input_text = f"summarize: {article}"
        
        # Tokenizer l'article (input)
        tokenized_input = t5_tokenizer(
            input_text,
            max_length=MAX_INPUT_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors=None
        )
        
        # Tokenizer le résumé (label)
        tokenized_label = t5_tokenizer(
            summary,
            max_length=MAX_TARGET_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors=None
        )
        
        input_ids.append(tokenized_input['input_ids'])
        attention_masks.append(tokenized_input['attention_mask'])
        labels.append(tokenized_label['input_ids'])
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    }

print(" Fonction de tokenization définie !")

# ─────────────────────────────────────────
# ÉTAPE 3 : TOKENIZER ET SAUVEGARDER (VERSION OPTIMISÉE)
# ─────────────────────────────────────────
print("\n3. TOKENIZATION ET SAUVEGARDE...")
print("    Utilisation de HuggingFace Datasets + sauvegarde par shards...")

# Créer le dossier de sortie
T5_DIR = PROCESSED_DIR / "t5"
T5_DIR.mkdir(parents=True, exist_ok=True)

def tokenize_and_save_t5(dataset, split_name, batch_size=2000):
    """
    Tokenizer et sauvegarder un dataset pour T5.
    
    Args:
        dataset: Dataset HuggingFace à tokenizer
        split_name: Nom du split (train_10, train_50, etc.)
        batch_size: Taille des batches de traitement
    """
    print(f"\n   • Traitement de {split_name}...")
    print(f"     Dataset: {len(dataset):,} exemples")
    
    # Tokenizer le dataset complet
    tokenized_dataset = dataset.map(
        tokenize_for_t5_batch,
        batched=True,
        batch_size=batch_size,
        remove_columns=dataset.column_names,
        desc=f"  Tokenization {split_name}"
    )
    
    # Sauvegarder avec HuggingFace (format Arrow optimisé)
    output_path = T5_DIR / split_name
    print(f"     Sauvegarde dans {output_path}...")
    
    # Sauvegarder avec num_shards pour éviter la saturation
    num_shards = max(1, len(tokenized_dataset) // 10000)  # 1 shard par 10k exemples
    tokenized_dataset.save_to_disk(output_path, num_shards=num_shards)
    
    print(f"     ✓ {split_name} sauvegardé: {len(tokenized_dataset):,} exemples")
    print(f"     ✓ Shards: {num_shards}")
    
    # Calculer la taille totale
    total_size = sum(f.stat().st_size for f in output_path.glob("*.arrow") if f.is_file())
    print(f"     ✓ Taille: {total_size / (1024*1024):.2f} MB")
    
    # Libérer la mémoire
    del tokenized_dataset
    gc.collect()
    
    print(f"     ✓ Mémoire libérée")

# Tokenizer et sauvegarder chaque split
try:
    print("\n" + "=" * 60)
    print("TOKENIZATION DES SPLITS")
    print("=" * 60)
    
    train_10_t5 = tokenize_and_save_t5(train_10, "train_10")
    print("  ✓ train_10 terminé")
    gc.collect()
    
    train_50_t5 = tokenize_and_save_t5(train_50, "train_50")
    print("  ✓ train_50 terminé")
    gc.collect()
    
    train_100_t5 = tokenize_and_save_t5(train_100, "train_100")
    print("  ✓ train_100 terminé")
    gc.collect()
    
    val_t5 = tokenize_and_save_t5(val_dataset_clean, "validation")
    print("  ✓ validation terminé")
    gc.collect()
    
    test_t5 = tokenize_and_save_t5(test_dataset_clean, "test")
    print("  ✓ test terminé")
    gc.collect()
    
    print("\n" + "=" * 60)
    print(" Tokenization et sauvegarde T5 terminées !")
    print("=" * 60)
    
except Exception as e:
    print(f"\n ERREUR: {e}")
    import traceback
    traceback.print_exc()

# ─────────────────────────────────────────
# ÉTAPE 4 : VÉRIFICATION
# ─────────────────────────────────────────
print("\n4. VÉRIFICATION...")
print("-" * 60)

from datasets import load_from_disk

# Vérifier les fichiers
for split_name in ["train_10", "train_50", "train_100", "validation", "test"]:
    output_path = T5_DIR / split_name
    
    if output_path.exists():
        try:
            # Charger le dataset
            ds = load_from_disk(output_path)
            
            # Calculer la taille totale
            total_size = sum(f.stat().st_size for f in output_path.glob("*.arrow") if f.is_file())
            
            # Compter les shards
            num_shards = len(list(output_path.glob("*.arrow")))
            
            print(f"   ✓ {split_name:15s}: {len(ds):>10,} exemples | {total_size / (1024*1024):>8.2f} MB | {num_shards} shards")
            
            # Vérifier les colonnes
            print(f"     Colonnes: {ds.column_names}")
            
            # Afficher un exemple
            example = ds[0]
            print(f"     Input IDs: {len(example['input_ids'])} tokens")
            print(f"     Attention mask: {len(example['attention_mask'])} tokens")
            print(f"     Labels: {len(example['labels'])} tokens")
            
            # Décoder pour vérifier
            decoded_input = t5_tokenizer.decode(example['input_ids'], skip_special_tokens=True)
            decoded_label = t5_tokenizer.decode([t for t in example['labels'] if t != -100], skip_special_tokens=True)
            print(f"     Input: {decoded_input[:100]}...")
            print(f"     Label: {decoded_label[:100]}...")
            
            # Libérer la mémoire
            del ds
            gc.collect()
            
        except Exception as e:
            print(f"    Erreur lors de la vérification de {split_name}: {e}")
    else:
        print(f"    {split_name:15s}: N'existe pas!")

print("\n" + "=" * 60)
print(" Données T5 prêtes et sauvegardées !")
print(f"\n Dossier: {T5_DIR}")
print("\n Structure des fichiers:")
for split_name in ["train_10", "train_50", "train_100", "validation", "test"]:
    output_path = T5_DIR / split_name
    if output_path.exists():
        shards = list(output_path.glob("*.arrow"))
        dataset_info = output_path / "dataset_info.json"
        print(f"  {split_name}/: {len(shards)} shards" + (", dataset_info.json" if dataset_info.exists() else ""))
print("=" * 60)


 PRÉPARATION DES DONNÉES POUR T5

1. CHARGEMENT DU TOKENIZER T5...
 Tokenizer T5 chargé !
   • Vocabulaire: 32,100 tokens
   • Modèle: t5-small

2. FONCTION DE TOKENIZATION...
 Fonction de tokenization définie !

3. TOKENIZATION ET SAUVEGARDE...
    Utilisation de HuggingFace Datasets + sauvegarde par shards...

TOKENIZATION DES SPLITS

   • Traitement de train_10...
     Dataset: 28,711 exemples


  Tokenization train_10:   0%|          | 0/28711 [00:00<?, ? examples/s]

     Sauvegarde dans data\processed\t5\train_10...


Saving the dataset (0/2 shards):   0%|          | 0/28711 [00:00<?, ? examples/s]

     ✓ train_10 sauvegardé: 28,711 exemples
     ✓ Shards: 2
     ✓ Taille: 84.46 MB
     ✓ Mémoire libérée
  ✓ train_10 terminé

   • Traitement de train_50...
     Dataset: 143,556 exemples


...

     Sauvegarde dans data\processed\t5\test...


Saving the dataset (0/1 shards):   0%|          | 0/11490 [00:00<?, ? examples/s]

     ✓ test sauvegardé: 11,490 exemples
     ✓ Shards: 1
     ✓ Taille: 33.80 MB
     ✓ Mémoire libérée
  ✓ test terminé

 Tokenization et sauvegarde T5 terminées !

4. VÉRIFICATION...
------------------------------------------------------------
   ✓ train_10       :     28,711 exemples |    84.46 MB | 2 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: summarize: Nasa has warned of an impending asteroid pass - and says it will be the closest until 202...
     Label: 2004 BL86 will pass about three times the distance of Earth to the moon. Estimate that the asteroid ...
   ✓ train_50       :    143,556 exemples |   422.29 MB | 14 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: summarize: (CNN) -- When convicted intelligence leaker Bradley Manning asked Thursday for hormone

Loading dataset from disk:   0%|          | 0/28 [00:00<?, ?it/s]

   ✓ train_100      :    287,113 exemples |   844.58 MB | 28 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: summarize: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reporte...
     Label: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday. Young actor says he has ...
   ✓ validation     :     13,368 exemples |    39.32 MB | 1 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']
     Input IDs: 512 tokens
     Attention mask: 512 tokens
     Labels: 64 tokens
     Input: summarize: (CNN)Share, and your gift will be multiplied. That may sound like an esoteric adage, but ...
     Label: Zully Broussard decided to give a kidney to a stranger. A new computer program helped her donation s...
   ✓ test           :     11,490 exemples |    33.80 MB | 1 shards
     Colonnes: ['input_ids', 'attention_mask', 'labels']